# Clean the HCP by care level and LGA table from Excel (ML-ready long format)

This notebook builds a simple data pipeline for the uploaded Excel file and reshapes the care level columns into an ML-ready long table.

## Cleaning rules
- read only the required worksheet
- detect the real header row from the Excel sheet
- keep only the required columns
- drop completely empty rows
- trim and normalize whitespace in text fields
- convert blank strings to missing values
- rename the columns for psql / analysis
- remove commas from level counts and totals and convert them to integers
- remove repeated header rows if they appear inside the data
- remove exact duplicate rows
- reshape `Level 1` to `Level 4` into a long format with `care_level` and `people_count`

At the end, the final cleaned dataframe is available as `clean_long_df`.


In [1]:
from pathlib import Path
import pandas as pd

INPUT_XLSX = Path("input/gen_data_home_care_demand_by_service_loc.xlsx")
SHEET_NAME = "Table 3.1 (LGA)"

INPUT_XLSX, SHEET_NAME

(PosixPath('input/gen_data_home_care_demand_by_service_loc.xlsx'),
 'Table 3.1 (LGA)')

In [2]:
raw_df = pd.read_excel(
    INPUT_XLSX,
    sheet_name=SHEET_NAME,
    header=None
)

raw_df.head(15)

,0,1,2,3,4,5,6
0,Table 3.1: Number of people using home care (H...,NaN,NaN,NaN,NaN,NaN,NaN
1,Return to contents,NaN,NaN,NaN,NaN,NaN,NaN
2,LGA code,LGA name,Level 1,Level 2,Level 3,Level 4,Total
3,10050,Albury,90,424,234,96,844
4,10180,Armidale,29,194,147,61,431
5,10250,Ballina,81,410,283,115,889
6,10470,Bathurst,13,134,156,93,396
7,10500,Bayside (NSW),30,298,225,121,674
8,10550,Bega Valley,37,409,261,73,780
9,10600,Bellingen,13,39,39,12,103


In [3]:
def clean_hcp_by_service_loc_long(input_path: Path, sheet_name: str = "Table 3.1"):
    # Read only the required worksheet without assuming the header row position.
    df = pd.read_excel(input_path, sheet_name=sheet_name, header=None)

    # Find the real header row by matching the first two header cells.
    col0 = df.iloc[:, 0].astype("string").str.strip()
    col1 = df.iloc[:, 1].astype("string").str.strip()

    header_mask = (
        col0.str.contains(r"^LGA code$", na=False)
        & col1.str.contains(r"^LGA name$", na=False)
    )

    if not header_mask.any():
        raise ValueError("Could not find the header row for the HCP service-location table.")

    header_row_idx = header_mask[header_mask].index[0]

    # Rebuild the dataframe using the detected header row.
    df = df.iloc[header_row_idx + 1:].copy()
    df.columns = [
        "lga_code_raw",
        "lga_name_raw",
        "level_1_raw",
        "level_2_raw",
        "level_3_raw",
        "level_4_raw",
        "total_raw",
    ]

    # Keep only the required columns.
    df = df[
        [
            "lga_code_raw",
            "lga_name_raw",
            "level_1_raw",
            "level_2_raw",
            "level_3_raw",
            "level_4_raw",
            "total_raw",
        ]
    ].copy()

    # Normalize whitespace and blanks.
    for col in df.columns:
        df[col] = df[col].astype("string").str.strip()

    df = df.replace(r"^\s*$", pd.NA, regex=True)

    # Remove repeated header rows that may appear inside the sheet.
    repeated_header_mask = (
        df["lga_code_raw"].astype("string").str.strip().eq("LGA code")
        & df["lga_name_raw"].astype("string").str.strip().eq("LGA name")
    )
    df = df.loc[~repeated_header_mask].copy()

    # Drop fully empty rows.
    df = df.dropna(how="all").copy()

    # Rename raw columns.
    df = df.rename(
        columns={
            "lga_code_raw": "lga_code",
            "lga_name_raw": "lga_name",
            "level_1_raw": "level_1",
            "level_2_raw": "level_2",
            "level_3_raw": "level_3",
            "level_4_raw": "level_4",
            "total_raw": "total",
        }
    )

    # Remove commas and convert numeric columns.
    numeric_cols = ["level_1", "level_2", "level_3", "level_4", "total"]
    for col in numeric_cols:
        df[col] = (
            df[col]
            .astype("string")
            .str.replace(",", "", regex=False)
            .str.strip()
        )
        df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")

    # Clean text columns.
    df["lga_code"] = df["lga_code"].astype("string").str.strip()
    df["lga_name"] = df["lga_name"].astype("string").str.strip()

    # Remove rows without essential identifiers.
    df = df.dropna(subset=["lga_code", "lga_name"]).copy()

    # Rename final wide columns for storage / analysis.
    clean_wide_df = df.rename(
        columns={
            "lga_code": "lga_code",
            "lga_name": "lga_name",
            "total": "total_people",
        }
    ).copy()

    # Remove exact duplicates in the wide table.
    clean_wide_df = clean_wide_df.drop_duplicates().reset_index(drop=True)

    # Reshape to long format for ML-ready downstream use.
    clean_long_df = clean_wide_df.melt(
        id_vars=["lga_code", "lga_name"],
        value_vars=["level_1", "level_2", "level_3", "level_4"],
        var_name="care_level_raw",
        value_name="people_count",
    )

    clean_long_df["care_level"] = (
        clean_long_df["care_level_raw"]
        .astype("string")
        .str.extract(r"(\d+)")
        .astype("Int64")
    )

    clean_long_df = clean_long_df.drop(columns=["care_level_raw"])

    # Drop rows with missing long-format essentials.
    clean_long_df = clean_long_df.dropna(
        subset=["lga_code", "lga_name", "care_level", "people_count"]
    ).copy()

    # Final column order.
    clean_long_df = clean_long_df[
        ["lga_code", "lga_name", "care_level", "people_count"]
    ].copy()

    # Remove exact duplicates in the long table.
    clean_long_df = clean_long_df.drop_duplicates().reset_index(drop=True)

    summary = {
        "header_row_index": int(header_row_idx),
        "wide_rows": int(len(clean_wide_df)),
        "long_rows": int(len(clean_long_df)),
        "null_lga_code_rows": int(clean_long_df["lga_code"].isna().sum()),
        "null_lga_name_rows": int(clean_long_df["lga_name"].isna().sum()),
        "null_care_level_rows": int(clean_long_df["care_level"].isna().sum()),
        "null_people_count_rows": int(clean_long_df["people_count"].isna().sum()),
        "duplicate_long_rows_removed": int(
            len(clean_long_df) - len(clean_long_df.drop_duplicates())
        ),
    }

    return clean_wide_df, clean_long_df, summary

In [4]:
clean_wide_df, clean_long_df, summary = clean_hcp_by_service_loc_long(INPUT_XLSX, SHEET_NAME)
summary

{'header_row_index': 2,
 'wide_rows': 324,
 'long_rows': 1296,
 'null_lga_code_rows': 0,
 'null_lga_name_rows': 0,
 'null_care_level_rows': 0,
 'null_people_count_rows': 0,
 'duplicate_long_rows_removed': 0}

In [5]:
clean_long_df.head(12)

,lga_code,lga_name,care_level,people_count
0,10050,Albury,1,90
1,10180,Armidale,1,29
2,10250,Ballina,1,81
3,10470,Bathurst,1,13
4,10500,Bayside (NSW),1,30
5,10550,Bega Valley,1,37
6,10600,Bellingen,1,13
7,10650,Berrigan,1,7
8,10750,Blacktown,1,70
9,10900,Blue Mountains,1,22


In [6]:
print(f"Blank rows remaining in long_df: {int(clean_long_df.isna().all(axis=1).sum())}")
print(f"Exact duplicates remaining in long_df: {int(clean_long_df.duplicated().sum())}")
print(f"Null lga_code rows: {int(clean_long_df['lga_code'].isna().sum())}")
print(f"Null lga_name rows: {int(clean_long_df['lga_name'].isna().sum())}")
print(f"Null care_level rows: {int(clean_long_df['care_level'].isna().sum())}")
print(f"Null people_count rows: {int(clean_long_df['people_count'].isna().sum())}")
print(f"Data types:\n{clean_long_df.dtypes}")

Blank rows remaining in long_df: 0
Exact duplicates remaining in long_df: 0
Null lga_code rows: 0
Null lga_name rows: 0
Null care_level rows: 0
Null people_count rows: 0
Data types:
lga_code        string
lga_name        string
care_level       Int64
people_count     Int64
dtype: object


# Insert Clean Long Data into DB

In [7]:
from pathlib import Path
import pandas as pd
from sqlalchemy import create_engine, text
import os
from dotenv import load_dotenv

# -----------------------------
# 1. DATABASE CONNECTION
# -----------------------------
load_dotenv('.env.db', override=True)

DB_USER = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_HOST = os.getenv("DB_HOST")
DB_PORT = os.getenv("DB_PORT")
DB_NAME = os.getenv("DB_NAME")

DATABASE_URL = (
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}"
    f"@{DB_HOST}:{DB_PORT}/{DB_NAME}?sslmode=require"
)

engine = create_engine(DATABASE_URL)

# -----------------------------
# 2. CREATE TABLE FROM SCHEMA
# -----------------------------
schema_path = Path("schemas/home_care_demand_by_service_lga.sql")

with open(schema_path, "r", encoding="utf-8") as f:
    schema_sql = f.read()

with engine.begin() as conn:
    conn.execute(text(schema_sql))

print("Schema created or already exists.")
print(DB_NAME)

Schema created or already exists.
carelink


In [8]:
# -----------------------------
# 3. PREPARE DATAFRAME
# -----------------------------
df_insert = clean_long_df.copy()

target_columns = [
    "lga_code",
    "lga_name",
    "care_level",
    "people_count",
]

df_insert = df_insert[target_columns].copy()

text_cols = [
    "lga_code",
    "lga_name",
]

for col in text_cols:
    df_insert[col] = df_insert[col].astype("string").str.strip()

df_insert["care_level"] = df_insert["care_level"].astype("Int64")
df_insert["people_count"] = pd.to_numeric(df_insert["people_count"], errors="coerce").astype("Int64")

df_insert.head()

,lga_code,lga_name,care_level,people_count
0,10050,Albury,1,90
1,10180,Armidale,1,29
2,10250,Ballina,1,81
3,10470,Bathurst,1,13
4,10500,Bayside (NSW),1,30


In [9]:
# -----------------------------
# 4. INSERT INTO DATABASE
# -----------------------------
df_insert.to_sql(
    "home_care_demand_by_service_lga",
    con=engine,
    if_exists="append",
    index=False,
    method="multi",
    chunksize=1000,
)

print(f"Inserted {len(df_insert)} rows into home_care_demand_by_service_lga")

# -----------------------------
# 5. VALIDATE
# -----------------------------
with engine.begin() as conn:
    result = conn.execute(text("SELECT COUNT(*) FROM home_care_demand_by_service_lga"))
    row_count = result.scalar()

print("Rows in table:", row_count)

Inserted 1296 rows into home_care_demand_by_service_lga
Rows in table: 1296
